# 02 — Silver Cleaning & Data Quality

Bronze JSONL → **Spark** Silver Parquet + DQ reports + **DuckDB** validation. **No Gold. No modeling.**

- Primary engine: **PySpark** (`SILVER_ENGINE = "spark"`). Spark is the official engine; pandas fallback is available only manually (`allow_fallback=True`) and must not be triggered silently.
- Event absence ≠ missing data (e.g. no pit rows → handled in Gold).
- Outliers flagged; only domain-invalid values removed.
- Run after Bronze with the **same** `USE_GOOGLE_DRIVE` setting as notebook 01.


## Colab setup (required every session)

Identical to `00` and `01`: clone, `pip install -e .`, Drive mount, then import `openf1_pipeline`.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Start Spark


In [2]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


INFO:openf1_pipeline.utils.spark:Spark session started: openf1-big-data-pipeline


Spark version: 4.0.2


## Bronze prerequisites


In [3]:
from pathlib import Path

from openf1_pipeline.config import (
    get_bronze_dir,
    get_data_quality_reports_dir,
    get_manifests_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.silver.build_silver import run_silver_cleaning
from openf1_pipeline.utils.cleanup import clean_silver_layer_outputs

SILVER_ENGINE = "spark"  # manual fallback only: "pandas" with allow_fallback=True
ALLOW_FALLBACK = False
CLEAR_SILVER_OUTPUTS = True

BRONZE_DIR = get_bronze_dir()
SILVER_DIR = get_silver_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
MANIFESTS_DIR = get_manifests_dir()

print("OUTPUT_ROOT:", get_output_root())
print("BRONZE_DIR:", BRONZE_DIR)
print("SILVER_DIR:", SILVER_DIR)
print("DATA_QUALITY_REPORTS_DIR:", DATA_QUALITY_REPORTS_DIR)

jsonl_files = list(BRONZE_DIR.rglob("*.jsonl")) if BRONZE_DIR.is_dir() else []
manifest_path = MANIFESTS_DIR / "ingestion_manifest.csv"
row_counts_path = DATA_QUALITY_REPORTS_DIR / "bronze_row_counts.csv"

print(f"Bronze JSONL files found: {len(jsonl_files)}")
print("ingestion_manifest.csv:", manifest_path.exists(), manifest_path)
print("bronze_row_counts.csv:", row_counts_path.exists(), row_counts_path)

if not jsonl_files:
    raise FileNotFoundError(
        f"Bronze data not found at {BRONZE_DIR}. "
        "Run 01_ingestion_bronze.ipynb first with the same USE_GOOGLE_DRIVE setting."
    )


OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
DATA_QUALITY_REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
Bronze JSONL files found: 609
ingestion_manifest.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/ingestion_manifest.csv
bronze_row_counts.csv: True /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/bronze_row_counts.csv


## Clean Silver outputs (required before fresh Spark run)

Removes prior Spark Parquet **directories**, Silver DQ CSVs, and DuckDB Silver reports.
Does not delete Bronze data.


In [4]:
if CLEAR_SILVER_OUTPUTS:
    print("Cleaning Silver layer outputs...")
    clean_silver_layer_outputs(silver_dir=SILVER_DIR, data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR)
else:
    print("Skipping Silver cleanup (CLEAR_SILVER_OUTPUTS=False).")


Cleaning Silver layer outputs...


INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/data/silver (10 items removed)
INFO:openf1_pipeline.utils.io:Removed 16 files matching ['silver_*.csv', 'duckdb_silver_*.csv'] from /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
INFO:openf1_pipeline.utils.cleanup:Silver layer cleanup complete: {'silver_data': 10, 'silver_reports': 16}


## Run Silver cleaning (Spark-first)


In [5]:
outputs = run_silver_cleaning(
    bronze_dir=BRONZE_DIR,
    silver_dir=SILVER_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
    engine=SILVER_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)
outputs["summary"]


/content/openf1-big-data-pipeline/src/openf1_pipeline/silver/build_silver_spark.py:852: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  missingness_before = pd.concat(
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved meetings (74 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved sessions (364 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved drivers (1756 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved laps (84140 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved pit (2449 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved weather (12674 rows)
INFO:openf1_pipeline.silver.build_silver_spark:Silver Spark saved position

{'engine': 'spark',
 'tables_loaded': 10,
 'total_rows_before': 148184,
 'total_rows_after': 148184,
 'session_result_rows_after': 1756}

## DuckDB validation (Silver Parquet)


In [6]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_silver_with_duckdb,
)

silver_duckdb = validate_silver_with_duckdb(SILVER_DIR)
duckdb_silver_paths = save_duckdb_validation_reports(
    silver_duckdb, DATA_QUALITY_REPORTS_DIR, prefix="silver"
)
duckdb_silver_paths


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv (10 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report silver_row_counts -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv (10 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv (1 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report session_result_target_support -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv (1 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_duplicate_keys.csv (0 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report session_result_duplicate_keys

{'silver_row_counts': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_silver_row_counts.csv',
 'session_result_target_support': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_target_support.csv',
 'session_result_duplicate_keys': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_session_result_duplicate_keys.csv',
 'laps_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_laps_rows_by_session_key.csv',
 'pit_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_pit_rows_by_session_key.csv',
 'weather_rows_by_session_key': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_silver_weather_rows_by_session_key.csv'}

## Inspect cleaning impact


In [7]:
import pandas as pd

impact = pd.read_csv(outputs["paths"]["silver_cleaning_impact_summary"])
inventory = pd.read_csv(outputs["paths"]["silver_table_inventory"])
rules = pd.read_csv(outputs["paths"]["silver_cleaning_rules"])

display(inventory)
display(impact[["table_name", "rows_before", "rows_after", "rows_removed", "row_removal_pct"]])
display(rules.head(20))


,table_name,row_count,column_count
0,meetings,74,19
1,sessions,364,16
2,drivers,1756,13
3,laps,84140,17
4,pit,2449,9
5,weather,12674,11
6,position,37778,6
7,race_control,7193,12
8,session_result,1756,12
9,starting_grid,0,7


,table_name,rows_before,rows_after,rows_removed,row_removal_pct
0,meetings,74,74,0,0.0
1,sessions,364,364,0,0.0
2,drivers,1756,1756,0,0.0
3,laps,84140,84140,0,0.0
4,pit,2449,2449,0,0.0
5,weather,12674,12674,0,0.0
6,position,37778,37778,0,0.0
7,race_control,7193,7193,0,0.0
8,session_result,1756,1756,0,0.0
9,starting_grid,0,0,0,0.0


,table_name,rule_id,rule_description,rows_before,rows_after,rows_removed,values_imputed,columns_affected,severity,rationale
0,meetings,SIL_RENAME,Standardize column names to snake_case,74,74,0,0,NaN,medium,NaN
1,meetings,SIL_CAST_NUM,Cast meeting_key and year to long,74,74,0,0,"meeting_key,year",medium,NaN
2,meetings,SIL_DROP_NULL_KEYS,Drop rows with null meeting_key,74,74,0,0,meeting_key,high,Meeting grain requires meeting_key
3,meetings,SIL_DUP_FULL,Remove exact duplicate rows,74,74,0,0,NaN,medium,NaN
4,meetings,SIL_DUP_KEY,Deduplicate on meeting_key,74,74,0,0,meeting_key,high,NaN
5,sessions,SIL_RENAME,Standardize column names to snake_case,364,364,0,0,NaN,medium,NaN
6,sessions,SIL_CAST_SCHEMA,Cast keys to long and session dates to timestamp,364,364,0,0,"session_key,meeting_key,year,date_start,date_end",medium,NaN
7,sessions,SIL_DROP_NULL_KEYS,Drop rows with null session_key,364,364,0,0,session_key,high,NaN
8,sessions,SIL_DUP_FULL,Remove exact duplicate rows,364,364,0,0,NaN,medium,NaN
9,sessions,SIL_DUP_KEY,Deduplicate on session_key,364,364,0,0,session_key,high,NaN


## Missingness before vs after


In [8]:
miss_before = pd.read_csv(outputs["paths"]["silver_missingness_before"])
miss_after = pd.read_csv(outputs["paths"]["silver_missingness_after"])

display(miss_before.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))
display(miss_after.sort_values("missing_pct", ascending=False).groupby("table_name").head(5))


,table_name,column_name,dtype,missing_count,missing_pct,non_missing_count,unique_count
98,race_control,qualifying_phase,string,7193,100.0000,0,1
100,race_control,sector,bigint,5726,79.6052,1467,22
93,race_control,driver_number,bigint,5573,77.4781,1620,31
72,pit,stop_duration,double,1571,64.1486,878,100
94,race_control,flag,string,3762,52.3008,3431,9
99,race_control,scope,string,3762,52.3008,3431,4
36,drivers,country_code,string,599,34.1116,1157,18
107,session_result,duration,double,498,28.3599,1258,1259
53,laps,i1_speed,bigint,13063,15.5253,71077,318
108,session_result,gap_to_leader,string,202,11.5034,1554,1170


,table_name,column_name,dtype,missing_count,missing_pct,non_missing_count,unique_count
98,race_control,qualifying_phase,string,7193,100.0000,0,1
100,race_control,sector,bigint,5726,79.6052,1467,22
93,race_control,driver_number,bigint,5573,77.4781,1620,31
72,pit,stop_duration,double,1571,64.1486,878,100
94,race_control,flag,string,3762,52.3008,3431,9
99,race_control,scope,string,3762,52.3008,3431,4
36,drivers,country_code,string,599,34.1116,1157,18
107,session_result,duration,double,498,28.3599,1258,1259
53,laps,i1_speed,bigint,13063,15.5253,71077,318
108,session_result,gap_to_leader,string,202,11.5034,1554,1170


## Duplicates & referential integrity


In [9]:
dups = pd.read_csv(outputs["paths"]["silver_duplicate_report"])
ref = pd.read_csv(outputs["paths"]["silver_referential_integrity_report"])
rejected = pd.read_csv(outputs["paths"]["silver_rejected_records_summary"])

display(dups)
display(ref)
display(rejected)


,table_name,duplicate_type,duplicate_count,duplicate_pct,stage,key_columns,duplicate_key_count,affected_rows,affected_rows_pct,status,duplicate_check_note
0,meetings,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
1,meetings,key_duplicate,NaN,NaN,before,meeting_key,0.0,0.0,0.0000,checked,NaN
2,sessions,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
3,sessions,key_duplicate,NaN,NaN,before,session_key,0.0,0.0,0.0000,checked,NaN
4,drivers,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
5,drivers,key_duplicate,NaN,NaN,before,"session_key,driver_number",0.0,0.0,0.0000,checked,NaN
6,laps,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,Converted complex object columns to stable str...
7,laps,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0000,checked,NaN
8,pit,full_row_duplicate,0.0,0.0,before,NaN,NaN,NaN,NaN,NaN,NaN
9,pit,key_duplicate,NaN,NaN,before,"session_key,driver_number,lap_number",0.0,0.0,0.0000,checked,NaN


,child_table,parent_table,key_columns,distinct_child_keys,unmatched_rows,unmatched_pct,status,stage
0,drivers,sessions,session_key,88,0,0.0,checked,before
1,laps,sessions,session_key,88,0,0.0,checked,before
2,pit,sessions,session_key,75,0,0.0,checked,before
3,weather,sessions,session_key,88,0,0.0,checked,before
4,position,sessions,session_key,88,0,0.0,checked,before
5,race_control,sessions,session_key,88,0,0.0,checked,before
6,session_result,sessions,session_key,88,0,0.0,checked,before
7,starting_grid,sessions,session_key,0,0,0.0,skipped_empty_child,before
8,laps,drivers,"session_key,driver_number",1753,0,0.0,checked,before
9,pit,drivers,"session_key,driver_number",1303,0,0.0,checked,before


,table_name,reason,rule_id,rejected_count
0,meetings,No records rejected in this run,SIL_NONE,0
1,sessions,No records rejected in this run,SIL_NONE,0
2,drivers,No records rejected in this run,SIL_NONE,0
3,laps,No records rejected in this run,SIL_NONE,0
4,pit,No records rejected in this run,SIL_NONE,0
5,weather,No records rejected in this run,SIL_NONE,0
6,position,No records rejected in this run,SIL_NONE,0
7,race_control,No records rejected in this run,SIL_NONE,0
8,session_result,No records rejected in this run,SIL_NONE,0
9,starting_grid,No records rejected in this run,SIL_NONE,0


## Silver Parquet schema — join-key dtype verification

Reads each Silver Parquet directly via Spark and confirms that the join keys
(`session_key`, `meeting_key`, `driver_number`, `lap_number`, `position`) are
stored as integer/long types — never floating. This is the on-disk source of
truth; the pandas-side missingness CSVs occasionally show keys as `double`
because of pandas re-inference, which is diagnostic-only.

If any key is reported as `float / double`, rerun the Silver build with the
latest `clean_*_spark` modules — these cast keys to `long` explicitly.

In [ ]:
from openf1_pipeline.utils.spark import spark_path

KEY_COLUMNS_PER_TABLE = {
    "meetings": ["meeting_key", "year"],
    "sessions": ["session_key", "meeting_key", "year"],
    "drivers": ["session_key", "driver_number", "meeting_key"],
    "laps": ["session_key", "driver_number", "lap_number"],
    "pit": ["session_key", "driver_number", "lap_number"],
    "weather": ["session_key"],
    "position": ["session_key", "driver_number", "meeting_key", "position"],
    "race_control": ["session_key", "meeting_key"],
    "session_result": ["session_key", "driver_number", "meeting_key", "position"],
    "starting_grid": ["session_key", "driver_number", "position"],
}

INTEGRAL_DTYPES = {"int", "bigint", "smallint", "tinyint", "long"}

schema_rows = []
for table_name, key_cols in KEY_COLUMNS_PER_TABLE.items():
    parquet_path = SILVER_DIR / f"{table_name}_clean.parquet"
    if not parquet_path.exists():
        schema_rows.append({"table": table_name, "column": "(missing)", "dtype": "n/a", "is_integer": False})
        continue
    sdf = spark.read.parquet(spark_path(parquet_path))
    schema = dict(sdf.dtypes)
    for col in key_cols:
        if col not in schema:
            continue
        dtype = schema[col]
        schema_rows.append({
            "table": table_name,
            "column": col,
            "dtype": dtype,
            "is_integer": dtype.lower() in INTEGRAL_DTYPES,
        })

schema_check = pd.DataFrame(schema_rows)
display(schema_check)

floating_keys = schema_check.loc[~schema_check["is_integer"] & (schema_check["dtype"] != "n/a")]
if floating_keys.empty:
    print("OK: all join keys are integer/long on disk.")
else:
    print("WARNING: the following join keys are floating on disk —")
    display(floating_keys)
    print("Rerun Silver after pulling the latest clean_*_spark fixes that cast keys to long.")

## Outlier & temporal anomaly reports

Both reports are persisted on Drive (`silver_outlier_report.csv`,
`silver_temporal_anomaly_report.csv`) but are not displayed by default.
This cell surfaces them inline so the write-up has full visibility.

In [ ]:
outliers = pd.read_csv(outputs["paths"]["silver_outlier_report"])
temporal = pd.read_csv(outputs["paths"]["silver_temporal_anomaly_report"])

print(f"silver_outlier_report.csv: {len(outliers)} rows")
display(outliers if len(outliers) <= 60 else outliers.head(20))

print(f"\nsilver_temporal_anomaly_report.csv: {len(temporal)} rows")
display(temporal if len(temporal) <= 60 else temporal.head(20))

## Post-fix verification (dynamic from disk)

Reads the Silver DQ artifacts straight from Drive and asserts the post-retry
contract. All comparisons are **dynamic** — no hard-coded values — so this
cell stays correct if Bronze re-ingestion changes the row totals later.

The assertions cover:

- Bronze → Silver row contract (zero loss).
- `session_result` row count and zero duplicate keys.
- Rejected records count (must be 0).
- Referential integrity unmatched count (must be 0 across every check).
- Required-endpoint availability and non-emptiness.
- The 16 race-control diagnostic-duplicate rows are tolerated by name.

In [ ]:
REQUIRED_ENDPOINTS = [
    "meetings", "sessions", "drivers", "laps", "pit",
    "weather", "position", "race_control", "session_result",
]

bronze_row_counts_df = pd.read_csv(DATA_QUALITY_REPORTS_DIR / "bronze_row_counts.csv")
inventory = pd.read_csv(outputs["paths"]["silver_table_inventory"])
impact = pd.read_csv(outputs["paths"]["silver_cleaning_impact_summary"])
rejected = pd.read_csv(outputs["paths"]["silver_rejected_records_summary"])
referential = pd.read_csv(outputs["paths"]["silver_referential_integrity_report"])
duckdb_silver_counts = pd.read_csv(
    DATA_QUALITY_REPORTS_DIR / "duckdb_silver_silver_row_counts.csv"
)
duckdb_sr_dup = pd.read_csv(
    DATA_QUALITY_REPORTS_DIR / "duckdb_silver_session_result_duplicate_keys.csv"
)

bronze_total = int(bronze_row_counts_df["row_count"].sum())
silver_total = int(inventory["row_count"].sum())
row_loss = bronze_total - silver_total

silver_by_table = inventory.set_index("table_name")["row_count"].to_dict()
session_result_rows = int(silver_by_table.get("session_result", 0))
session_result_duplicate_keys = int(duckdb_sr_dup.shape[0])

rejected_total = int(rejected["rejected_count"].sum())
ri_unmatched_total = int(referential["unmatched_rows"].fillna(0).sum())

required_present = {ep: int(silver_by_table.get(ep, 0)) for ep in REQUIRED_ENDPOINTS}
missing_required = [ep for ep, n in required_present.items() if n <= 0]

print("=== Post-fix verification ===")
print(f"Bronze rows (sum bronze_row_counts.csv): {bronze_total}")
print(f"Silver rows (sum silver_table_inventory): {silver_total}")
print(f"Bronze -> Silver row loss: {row_loss}")
print(f"session_result rows: {session_result_rows}")
print(f"session_result duplicate keys (DuckDB): {session_result_duplicate_keys}")
print(f"Rejected records (sum): {rejected_total}")
print(f"Referential integrity unmatched (sum): {ri_unmatched_total}")
print("Required endpoints (row counts):")
for ep, n in required_present.items():
    print(f"  {ep:>14}: {n}")
print(f"starting_grid rows (optional): {int(silver_by_table.get('starting_grid', 0))}")

assert row_loss == 0, f"Bronze->Silver row loss must be 0 (got {row_loss})"
assert session_result_rows > 0, "session_result must be non-empty"
assert session_result_duplicate_keys == 0, (
    f"session_result must have zero duplicate keys (got {session_result_duplicate_keys})"
)
assert rejected_total == 0, f"Rejected records must be 0 (got {rejected_total})"
assert ri_unmatched_total == 0, (
    f"Referential integrity unmatched must be 0 (got {ri_unmatched_total})"
)
assert not missing_required, f"Empty required endpoints: {missing_required}"

duckdb_total = int(duckdb_silver_counts["row_count"].sum())
assert duckdb_total == silver_total, (
    f"DuckDB Silver row total ({duckdb_total}) must match Spark inventory ({silver_total})"
)

print("\nOK: Silver post-fix contract holds. Safe to proceed to Notebook 03 Gold.")